In [ ]:
import numpy as np
import pandas as pd

# Use latest summary CSV produced by the backtest run
df = pd.read_csv(r"backtest_output/metrics/SUMMARY_BTCUSDT_20251016_204833.csv")

num_cols = [
    "rows",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "auc",
    "best_threshold",
    "trades",
    "hit_rate",
    "avg_net_ret_per_bar",
    "avg_net_ret_per_trade",
    "total_net_return",
    "sharpe_like",
    "cost_roundtrip",
    "r2",
    "mae_bps",
    "mape_bps",
]
for c in num_cols:
    if c not in df.columns:
        df[c] = np.nan
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["task"] = np.where(df["r2"].notna(), "regress", "classify")
df.head()

<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Chan\AppData\Local\Temp\ipykernel_26924\3047886526.py:4: SyntaxWarning: invalid escape sequence '\m'
  df = pd.read_csv("backtest_output\metrics\SUMMARY_BTCUSDT_20251011_204311_regress.csv")


,symbol,interval,start_str,timelag,model,rows,split_mode,test_size,class_weight,label_mode,...,total_net_return,sharpe_like,sortino_like,cost_roundtrip,r2,mae_bps,rmse_bps,mape_pct,mape_bps,task
0,BTCUSDT,4h,120 days ago UTC,16,linreg,673,time,0.2,NaN,return_bps,...,3.627312,0.056843,1.813656e+09,0.002,-0.266913,59.153500,84.432944,245.593580,NaN,regress
1,BTCUSDT,4h,120 days ago UTC,16,linreg,673,time,0.2,NaN,return_bps,...,3.627312,0.056843,1.813656e+09,0.002,-0.266913,59.153500,84.432944,245.593580,NaN,regress
2,BTCUSDT,4h,120 days ago UTC,16,rf_reg,673,time,0.2,NaN,return_bps,...,-63.311911,-0.068169,-2.090312e-01,0.002,-0.861592,82.752745,102.350516,439.659163,NaN,regress
3,BTCUSDT,4h,120 days ago UTC,16,hgb_reg,673,time,0.2,NaN,return_bps,...,-139.162278,-0.516139,-9.679382e-01,0.002,-0.321887,63.413217,86.250167,292.402076,NaN,regress
4,BTCUSDT,4h,120 days ago UTC,16,svr,673,time,0.2,NaN,return_bps,...,0.000000,NaN,NaN,0.002,-0.005034,50.960177,75.206092,103.790068,NaN,regress


In [2]:
MIN_ROWS = 200  # drop tiny runs
df1 = df.copy()

df1 = df1.loc[df1["rows"] >= MIN_ROWS]
df1 = df1.loc[df1["trades"].fillna(0) > 0]
df1 = df1.loc[~((df1["task"] == "regress") & (df1["r2"].isna()))]

print(f"After sanity filters: {len(df)} -> {len(df1)} rows")
df1.head()

After sanity filters: 108 -> 81 rows


,symbol,interval,start_str,timelag,model,rows,split_mode,test_size,class_weight,label_mode,...,total_net_return,sharpe_like,sortino_like,cost_roundtrip,r2,mae_bps,rmse_bps,mape_pct,mape_bps,task
0,BTCUSDT,4h,120 days ago UTC,16,linreg,673,time,0.2,NaN,return_bps,...,3.627312,0.056843,1.813656e+09,0.002,-0.266913,59.153500,84.432944,245.593580,NaN,regress
1,BTCUSDT,4h,120 days ago UTC,16,linreg,673,time,0.2,NaN,return_bps,...,3.627312,0.056843,1.813656e+09,0.002,-0.266913,59.153500,84.432944,245.593580,NaN,regress
2,BTCUSDT,4h,120 days ago UTC,16,rf_reg,673,time,0.2,NaN,return_bps,...,-63.311911,-0.068169,-2.090312e-01,0.002,-0.861592,82.752745,102.350516,439.659163,NaN,regress
3,BTCUSDT,4h,120 days ago UTC,16,hgb_reg,673,time,0.2,NaN,return_bps,...,-139.162278,-0.516139,-9.679382e-01,0.002,-0.321887,63.413217,86.250167,292.402076,NaN,regress
5,BTCUSDT,4h,120 days ago UTC,16,hgb_reg,673,time,0.2,NaN,return_bps,...,-139.162278,-0.516139,-9.679382e-01,0.002,-0.321887,63.413217,86.250167,292.402076,NaN,regress


In [3]:
MIN_SHARPE = 0.3
df2 = df1.copy()

df2 = df2.loc[df2["total_net_return"] > 0]
df2 = df2.loc[df2["avg_net_ret_per_bar"] > 0]
df2 = df2.loc[df2["sharpe_like"] > MIN_SHARPE]

print(f"After econ filters: {len(df1)} -> {len(df2)} rows")
df2.head()

After econ filters: 81 -> 26 rows


,symbol,interval,start_str,timelag,model,rows,split_mode,test_size,class_weight,label_mode,...,total_net_return,sharpe_like,sortino_like,cost_roundtrip,r2,mae_bps,rmse_bps,mape_pct,mape_bps,task
14,BTCUSDT,8h,120 days ago UTC,16,rf_reg,313,time,0.2,NaN,return_bps,...,397.761442,0.471761,4.020935,0.002,-0.249953,102.227530,134.656585,403.415654,NaN,regress
15,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,421.693818,0.356751,1.653156,0.002,-0.137016,91.837073,128.502756,369.406969,NaN,regress
17,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.905595,0.358638,1.673587,0.002,-0.136383,91.921134,128.592540,369.415473,NaN,regress
18,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.941832,0.358651,1.673729,0.002,-0.136379,91.921719,128.593176,369.415530,NaN,regress
19,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.644689,0.358546,1.672564,0.002,-0.136415,91.916926,128.587966,369.415060,NaN,regress


In [4]:
# --- 3) Regression-specific filters ----------------------------------------
df3 = df2.copy()
is_reg = df3["task"] == "regress"

# Drop bad regressors
df3 = df3.loc[~(is_reg & (df3["r2"] <= -0.5))]

# Drop huge MAE outliers (3× median per interval)
for iv, sub in df3.groupby("interval"):
    med = np.nanmedian(sub["mae_bps"])
    if not np.isnan(med) and med > 0:
        bad = (df3["interval"] == iv) & (df3["mae_bps"] > 3 * med)
        df3 = df3.loc[~bad]

print(f"After regression quality filters: {len(df2)} -> {len(df3)} rows")
df3.head()

After regression quality filters: 26 -> 26 rows


,symbol,interval,start_str,timelag,model,rows,split_mode,test_size,class_weight,label_mode,...,total_net_return,sharpe_like,sortino_like,cost_roundtrip,r2,mae_bps,rmse_bps,mape_pct,mape_bps,task
14,BTCUSDT,8h,120 days ago UTC,16,rf_reg,313,time,0.2,NaN,return_bps,...,397.761442,0.471761,4.020935,0.002,-0.249953,102.227530,134.656585,403.415654,NaN,regress
15,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,421.693818,0.356751,1.653156,0.002,-0.137016,91.837073,128.502756,369.406969,NaN,regress
17,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.905595,0.358638,1.673587,0.002,-0.136383,91.921134,128.592540,369.415473,NaN,regress
18,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.941832,0.358651,1.673729,0.002,-0.136379,91.921719,128.593176,369.415530,NaN,regress
19,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.644689,0.358546,1.672564,0.002,-0.136415,91.916926,128.587966,369.415060,NaN,regress


In [5]:
df4 = df3.copy()

stab = df4.groupby(["interval", "model"], as_index=False).agg(
    n_runs=("sharpe_like", "size"),
    median_sharpe=("sharpe_like", "median"),
    hit_med=("hit_rate", "median"),
    net_med=("total_net_return", "median"),
)

stab_keep = stab.loc[
    (stab["median_sharpe"] > 0) & (stab["net_med"] > 0), ["interval", "model"]
]
df5 = df4.merge(stab_keep, on=["interval", "model"], how="inner")

print(f"After stability filter: {len(df4)} -> {len(df5)} rows")
stab.sort_values(["interval", "median_sharpe"], ascending=[True, False]).head(10)

After stability filter: 26 -> 26 rows


,interval,model,n_runs,median_sharpe,hit_med,net_med
0,1d,arima,1,3.831583e+10,NaN,38.315834
3,1d,rf_reg,1,1.319538e+00,NaN,749.747863
2,1d,linreg,2,5.549687e-01,NaN,736.133673
1,1d,hgb_reg,7,5.514134e-01,NaN,838.426664
4,4h,linreg,4,7.585391e+10,NaN,120.670699
5,4h,rf_reg,1,8.222738e-01,NaN,136.911768
7,8h,linreg,2,1.524425e+11,NaN,152.442517
8,8h,rf_reg,1,4.717605e-01,NaN,397.761442
6,8h,hgb_reg,7,3.583491e-01,NaN,426.086641


In [6]:
# --- 6) Minimum activity & cost sanity ----------------
df6 = df5.copy()

# 1) Enforce minimum trades
MIN_TRADES = 5
df6 = df6.loc[(df6["trades"].fillna(0).astype(int) >= MIN_TRADES)]

# Prefer explicit avg_net_ret_per_trade if present & looks like bps already.
avg_trade_bps = df6.get("avg_net_ret_per_trade")

# If avg_net_ret_per_trade is missing or mostly NaN, derive from totals
if (avg_trade_bps is None) or (avg_trade_bps.isna().mean() > 0.3):
    avg_trade_bps = pd.Series(np.nan, index=df6.index)

# Fill from total / trades where possible (avoid divide-by-zero)
has_trades = df6["trades"] > 0
avg_trade_bps = avg_trade_bps.where(
    ~avg_trade_bps.isna(),
    df6["total_net_return"] / df6["trades"].where(has_trades, np.nan),
)

# 3) Convert cost to bps for fair comparison
# cost_roundtrip is a fraction (e.g., 0.0026 = 26 bps)
cost_bps = df6["cost_roundtrip"] * 10_000.0

# 4) Pick a mild safety margin (e.g., +5%) and apply cost sanity in bps
COST_MULTIPLIER = 1.05
ok_cost = avg_trade_bps >= (COST_MULTIPLIER * cost_bps)

# If avg_trade_bps is still NaN for a row, treat as fail
ok_cost = ok_cost.fillna(False)

before = len(df6)
df6 = df6.loc[ok_cost]

print(f"After activity & cost filter: {before} -> {len(df6)} rows")
df6.head()

After activity & cost filter: 17 -> 17 rows


,symbol,interval,start_str,timelag,model,rows,split_mode,test_size,class_weight,label_mode,...,total_net_return,sharpe_like,sortino_like,cost_roundtrip,r2,mae_bps,rmse_bps,mape_pct,mape_bps,task
0,BTCUSDT,8h,120 days ago UTC,16,rf_reg,313,time,0.2,NaN,return_bps,...,397.761442,0.471761,4.020935,0.002,-0.249953,102.227530,134.656585,403.415654,NaN,regress
1,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,421.693818,0.356751,1.653156,0.002,-0.137016,91.837073,128.502756,369.406969,NaN,regress
2,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.905595,0.358638,1.673587,0.002,-0.136383,91.921134,128.592540,369.415473,NaN,regress
3,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.941832,0.358651,1.673729,0.002,-0.136379,91.921719,128.593176,369.415530,NaN,regress
4,BTCUSDT,8h,120 days ago UTC,16,hgb_reg,313,time,0.2,NaN,return_bps,...,426.644689,0.358546,1.672564,0.002,-0.136415,91.916926,128.587966,369.415060,NaN,regress


In [7]:
# --- 7) Composite score per interval ---------------------------------------

df7 = df6.copy()


def rank_within_interval(g):
    for c in ["sharpe_like", "total_net_return", "avg_net_ret_per_bar", "trades"]:
        g[c + "_z"] = (g[c] - g[c].mean()) / (g[c].std(ddof=0) + 1e-9)
    g = g.merge(
        stab[["interval", "model", "n_runs", "median_sharpe"]],
        on=["interval", "model"],
        how="left",
    )
    g["n_runs"] = g["n_runs"].fillna(1)
    g["med_sharpe_z"] = (g["median_sharpe"] - g["median_sharpe"].mean()) / (
        g["median_sharpe"].std(ddof=0) + 1e-9
    )
    g["score"] = (
        0.45 * g["sharpe_like_z"]
        + 0.25 * g["total_net_return_z"]
        + 0.15 * g["avg_net_ret_per_bar_z"]
        + 0.10 * g["trades_z"]
        + 0.05 * g["med_sharpe_z"]
    )
    return g


df7 = df7.groupby("interval", group_keys=False).apply(rank_within_interval)
df7 = df7.sort_values(["interval", "score"], ascending=[True, False])
df7[
    [
        "interval",
        "model",
        "start_str",
        "task",
        "rows",
        "trades",
        "sharpe_like",
        "total_net_return",
        "avg_net_ret_per_bar",
        "score",
    ]
].head(20)

C:\Users\Chan\AppData\Local\Temp\ipykernel_26924\2342964610.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df7 = df7.groupby("interval", group_keys=False).apply(rank_within_interval)


,interval,model,start_str,task,rows,trades,sharpe_like,total_net_return,avg_net_ret_per_bar,score
7,1d,linreg,720 days ago UTC,regress,673,13,0.554969,736.133673,56.625667,0.374165
8,1d,linreg,720 days ago UTC,regress,673,13,0.554969,736.133673,56.625667,0.374165
0,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
1,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
2,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
3,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
4,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
5,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
6,1d,hgb_reg,365 days ago UTC,regress,318,12,0.551413,838.426664,69.868889,-0.106904
0,8h,rf_reg,120 days ago UTC,regress,313,10,0.471761,397.761442,39.776144,0.818593


In [8]:
# --- 8) Pick winners --------------------------------------------------------
TOPK = 3

winners = (
    df7.sort_values(["interval", "score"], ascending=[True, False])
    .groupby("interval")
    .head(TOPK)
    .reset_index(drop=True)
)

most_stable = (
    df7.sort_values(["interval", "median_sharpe"], ascending=[True, False])
    .groupby("interval")
    .head(1)
    .reset_index(drop=True)
)

print("Top-K by composite score:")
display(
    winners[
        [
            "interval",
            "model",
            "task",
            "start_str",
            "rows",
            "trades",
            "sharpe_like",
            "total_net_return",
            "avg_net_ret_per_bar",
            "score",
        ]
    ]
)

print("\nMost stable per interval (by median Sharpe):")
display(
    most_stable[
        [
            "interval",
            "model",
            "task",
            "rows",
            "trades",
            "median_sharpe",
            "total_net_return",
            "avg_net_ret_per_bar",
        ]
    ]
)

Top-K by composite score:


,interval,model,task,start_str,rows,trades,sharpe_like,total_net_return,avg_net_ret_per_bar,score
0,1d,linreg,regress,720 days ago UTC,673,13,0.554969,736.133673,56.625667,0.374165
1,1d,linreg,regress,720 days ago UTC,673,13,0.554969,736.133673,56.625667,0.374165
2,1d,hgb_reg,regress,365 days ago UTC,318,12,0.551413,838.426664,69.868889,-0.106904
3,8h,rf_reg,regress,120 days ago UTC,313,10,0.471761,397.761442,39.776144,0.818593
4,8h,hgb_reg,regress,120 days ago UTC,313,16,0.358651,426.941832,26.683865,-0.044444
5,8h,hgb_reg,regress,120 days ago UTC,313,16,0.358638,426.905595,26.681600,-0.045649



Most stable per interval (by median Sharpe):


,interval,model,task,rows,trades,median_sharpe,total_net_return,avg_net_ret_per_bar
0,1d,linreg,regress,673,13,0.554969,736.133673,56.625667
1,8h,rf_reg,regress,313,10,0.471761,397.761442,39.776144


In [9]:
# --- 9) Export shortlist ----------------------------------------------------
out_dir = "backtest_output/metrics"
winners.to_csv(f"{out_dir}/SHORTLIST_topk.csv", index=False)
most_stable.to_csv(f"{out_dir}/SHORTLIST_stable.csv", index=False)

print("Saved SHORTLIST_topk.csv and SHORTLIST_stable.csv")

Saved SHORTLIST_topk.csv and SHORTLIST_stable.csv
